In [ ]:
import sys
from pathlib import Path

# same reason as in 1-processing
sys.path.append(str(Path("..").resolve()))

# nest_asnycio instance to be able to call stan.build inside running event loop (Jupyter Notebooks)
import nest_asyncio

nest_asyncio.apply()

import pickle
import stan
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER
import numpy as np

import arviz as az

# Load processed data
data_path = PROCESSED_DATA_FOLDER / "processed_data.pkl"
model_data = ModelData.from_pickle(data_path)

print(model_data.summary())

In [ ]:
from constants import STAN_MODEL_FOLDER, FITTED_MODEL_FOLDER

# Konfiguration für Stan-Modelle
stan_model_folder = STAN_MODEL_FOLDER
fitted_model_folder = FITTED_MODEL_FOLDER
fitted_model_folder.mkdir(parents=True, exist_ok=True)

# Cache für kompilierte Modelle (beschleunigt das Training)
# In PyStan 3 gibt es kein separates StanModel-Objekt mehr,
# daher speichern wir den model_code für jedes Modell
model_code_cache = {}

print(f"✓ Configuration set")
print(f"  Stan models: {stan_model_folder}")
print(f"  Output folder: {fitted_model_folder}")

In [104]:
# Funktion zum Konvertieren von ModelData zu Stan-Format
def prepare_stan_data(model_data: ModelData, S: int):
    """
    Bereitet die Daten für Stan vor und fügt die Anzahl der States hinzu.

    Parameters:
    -----------
    model_data : ModelData
        ModelData Objekt mit allen Daten
    S : int
        Anzahl der Hidden States

    Returns:
    --------
    dict : Dictionary im Stan-Format
    """
    # Stan erwartet spezifische Variablennamen (Großbuchstaben)
    stan_data = {
        "S": S,
        "N_total": model_data.n_total,
        "N_train": model_data.n_train,
        "N_obs": model_data.n_obs,
        "nCovs": model_data.n_covs,
        "Time": model_data.time,
        "Closed": model_data.closed.astype(int).tolist(),
        "Days": model_data.days,
        "Ratings": model_data.ratings,
        "Sentiment": model_data.sentiment,
        "Q": model_data.Q.tolist(),
        "R": model_data.R.tolist(),
        "X_test": model_data.X_test.tolist(),
    }

    return stan_data


# Test mit S=3
test_data = prepare_stan_data(model_data, S=3)
print(f"✓ Stan data prepared")
print(f"  Keys: {list(test_data.keys())}")

✓ Stan data prepared
  Keys: ['S', 'N_total', 'N_train', 'N_obs', 'nCovs', 'Time', 'Closed', 'Days', 'Ratings', 'Sentiment', 'Q', 'R', 'X_test']


In [105]:
# Funktion zum Trainieren eines einzelnen Modells mit PyStan 3
def train_model(
    model_data,
    S,
    model_name="vdhmm",
    num_chains=2,
    num_samples=1000,
    num_warmup=None,
    seed=None,
):
    """
    Trainiert ein VD-HMM oder HMM Modell mit PyStan 3.

    Parameters:
    -----------
    model_data : ModelData
        ModelData Objekt mit den Daten
    S : int
        Anzahl der Hidden States (1-4)
    model_name : str
        'vdhmm' oder 'hmm'
    num_chains : int
        Anzahl der MCMC Chains
    num_samples : int
        Anzahl der Post-Warmup Samples pro Chain
    num_warmup : int, optional
        Anzahl der Warmup Iterationen (default: num_samples)
    seed : int, optional
        Random seed für Reproduzierbarkeit

    Returns:
    --------
    stan.fit.Fit : Das trainierte Modell (Fit-Objekt)
    """

    assert model_name in [
        "vdhmm",
        "hmm",
    ], f"Invalid model_name '{model_name}'. Must be 'vdhmm' or 'hmm'."

    assert S in range(
        2, 5 + 1
    ), "Number of states must be in the range between 2 and 5!"

    # Daten vorbereiten
    stan_data = prepare_stan_data(model_data, S)

    model_file = stan_model_folder / f"{model_name}.stan"

    if not model_file.exists():
        raise FileNotFoundError(f"Stan model file not found: {model_file}")

    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} with S={S} states")
    print(f"{'='*60}")
    print(f"Model file: {model_file}")

    # Warmup setzen (PyStan 3 default ist num_samples wenn nicht angegeben)
    if num_warmup is None:
        num_warmup = num_samples

    # Model Code laden (mit Caching)
    model_key = f"{model_name}_{S}"
    if model_key not in model_code_cache:
        print(f"Loading Stan model code...")
        with open(model_file, "r") as f:
            model_code_cache[model_key] = f.read()
    else:
        print("Using cached model code")

    model_code = model_code_cache[model_key]

    # MCMC Sampling mit PyStan 3 API
    print(f"\nBuilding and sampling model:")
    print(f"  Chains: {num_chains}")
    print(f"  Warmup iterations: {num_warmup}")
    print(f"  Sampling iterations: {num_samples}")
    print(f"  Seed: {seed if seed else 'random'}")

    # stan.build() kompiliert das Modell mit Daten und seed
    # Wichtig: In PyStan 3 wird data und random_seed bei build() übergeben
    posterior = stan.build(program_code=model_code, data=stan_data, random_seed=seed)

    # posterior.sample() zieht Samples

    init_values = [{}] * num_chains

    fit = posterior.sample(
        num_chains=num_chains,
        num_samples=num_samples,
        num_warmup=num_warmup,
        init=init_values,
    )

    # Modell speichern
    output_path = fitted_model_folder / f"{model_name}_{S}.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(
            {
                "fit": fit,
                "model_name": model_name,
                "S": S,
                "stan_data": stan_data,
                "model_code": model_code,
            },
            f,
        )

    print(f"\n✓ Model saved to {output_path}")

    # Diagnostics ausgeben
    print(f"\n{'-'*60}")
    print("Model Summary:")
    print(f"{'-'*60}")

    # In PyStan 3: Zugriff auf Parameter via fit['param_name']
    # Konvertiere zu DataFrame für schönere Ausgabe
    df = fit.to_frame()
    print(f"\nSampled parameters: {df.shape[1]} parameters, {df.shape[0]} draws")
    print(f"Parameter names (first 10): {list(df.columns[:10])}")

    # Basis-Statistiken
    print(f"\nBasic statistics:")
    print(df.describe())

    return fit


print("✓ Training function defined (PyStan 3 API)")

✓ Training function defined (PyStan 3 API)


## Training VD-HMM Models (S=2 bis S=4)

Variable-Duration Hidden Markov Models mit zeitabhängigen Übergangswahrscheinlichkeiten.


In [106]:
# Training Settings
SEED = 42  # Für Reproduzierbarkeit
NUM_CHAINS = 1
NUM_SAMPLES = 100  # Post-warmup samples pro chain
NUM_WARMUP = 100  # Warmup iterations pro chain

np.random.seed(42)

# Dictionary zum Speichern aller trainierten Modelle
trained_models = {}

print("Training Configuration:")
print(f"  Seed: {SEED}")
print(f"  Chains: {NUM_CHAINS}")
print(f"  Samples (post-warmup): {NUM_SAMPLES}")
print(f"  Warmup iterations: {NUM_WARMUP}")
print(f"  Total iterations per chain: {NUM_WARMUP + NUM_SAMPLES}")

Training Configuration:
  Seed: 42
  Chains: 1
  Samples (post-warmup): 100
  Warmup iterations: 100
  Total iterations per chain: 200


In [107]:
# VD-HMM Training für S=2 bis S=4
# WARNUNG: Dies kann mehrere Stunden dauern!

for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# VD-HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model(
            model_data=model_data,
            S=S,
            model_name="vdhmm",
            num_chains=NUM_CHAINS,
            num_samples=NUM_SAMPLES,
            num_warmup=NUM_WARMUP,
            seed=SEED,
        )

        trained_models[f"vdhmm_{S}"] = fit
        print(f"\n✓✓✓ VD-HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training VD-HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("VD-HMM Training Complete!")
print("=" * 60)



############################################################
# VD-HMM Training: S=2
############################################################


Training VDHMM with S=2 states
Model file: ../data/stan_code/vdhmm.stan
Loading Stan model code...

Building and sampling model:
  Chains: 1
  Warmup iterations: 100
  Sampling iterations: 100
  Seed: 42
Building...



Building: found in cache, done.Messages from stanc:
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_s27kz_r5/model_ltzbdsbj.stan', line 178, column 41: The
    variable state_emission may not have been assigned a value before its
    use.
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_s27kz_r5/model_ltzbdsbj.stan', line 132, column 33: The
    variable pos_obs may not have been assigned a value before its use.
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/200)
Sampling:  50% (100/200)
Sampling:  50% (101/200)
Sampling: 100% (200/200)
Sampling: 100% (200/200), done.
Messages received during sampling:
  Gradient evaluation took 0.086213 seconds
  1000 transitions using 10 leapfrog steps per transition would take 862.13 seconds.
  Adjust your expectations accordingly!
           three stages of adaptation as currently configured.
           Reducing


✓ Model saved to ../models/vdhmm_2.pkl

------------------------------------------------------------
Model Summary:
------------------------------------------------------------

Sampled parameters: 2923 parameters, 100 draws
Parameter names (first 10): ['lp__', 'accept_stat__', 'stepsize__', 'treedepth__', 'n_leapfrog__', 'divergent__', 'energy__', 'pi.1', 'pi.2', 'tpm.1.1']

Basic statistics:


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in s

parameters          lp__  accept_stat__    stepsize__  treedepth__  \
count         100.000000     100.000000  1.000000e+02   100.000000   
mean       -46491.951966       0.912784  6.046267e-03     6.990000   
std             4.495111       0.105086  8.717313e-19     1.150274   
min        -46502.132587       0.462261  6.046267e-03     6.000000   
25%        -46494.826649       0.876867  6.046267e-03     6.000000   
50%        -46492.190035       0.940660  6.046267e-03     7.000000   
75%        -46488.578082       0.990549  6.046267e-03     8.000000   
max        -46483.378332       1.000000  6.046267e-03    10.000000   

parameters  n_leapfrog__  divergent__      energy__        pi.1        pi.2  \
count          100.00000        100.0    100.000000  100.000000  100.000000   
mean           263.96000          0.0  46508.121814    0.315817    0.684183   
std            264.74283          0.0      5.725948    0.034951    0.034951   
min             63.00000          0.0  46496.422453  

...

parameters          lp__  accept_stat__    stepsize__  treedepth__  \
count         100.000000     100.000000  1.000000e+02   100.000000   
mean       -45951.615040       0.891280  1.209295e-02     7.580000   
std             4.207937       0.102926  1.743463e-18     0.944736   
min        -45961.879042       0.432812  1.209295e-02     5.000000   
25%        -45954.929414       0.841261  1.209295e-02     7.000000   
50%        -45951.087777       0.918583  1.209295e-02     8.000000   
75%        -45948.474452       0.976421  1.209295e-02     8.000000   
max        -45941.907673       1.000000  1.209295e-02     9.000000   

parameters  n_leapfrog__  divergent__      energy__        pi.1        pi.2  \
count         100.000000        100.0    100.000000  100.000000  100.000000   
mean          310.040000          0.0  45972.247642    0.095019    0.499901   
std           173.198194          0.0      6.189607    0.024650    0.048179   
min            31.000000          0.0  45960.478171  


Building: found in cache, done.Messages from stanc:
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_s27kz_r5/model_ltzbdsbj.stan', line 178, column 41: The
    variable state_emission may not have been assigned a value before its
    use.
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_s27kz_r5/model_ltzbdsbj.stan', line 132, column 33: The
    variable pos_obs may not have been assigned a value before its use.
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/200)
Sampling:  50% (100/200)
Sampling:  50% (101/200)
Sampling: 100% (200/200)
Sampling: 100% (200/200), done.
Messages received during sampling:
  Gradient evaluation took 0.964632 seconds
  1000 transitions using 10 leapfrog steps per transition would take 9646.32 seconds.
  Adjust your expectations accordingly!
           three stages of adaptation as currently configured.
           Reducin


✓ Model saved to ../models/vdhmm_4.pkl

------------------------------------------------------------
Model Summary:
------------------------------------------------------------

Sampled parameters: 3973 parameters, 100 draws
Parameter names (first 10): ['lp__', 'accept_stat__', 'stepsize__', 'treedepth__', 'n_leapfrog__', 'divergent__', 'energy__', 'pi.1', 'pi.2', 'pi.3']

Basic statistics:


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in s

parameters          lp__  accept_stat__    stepsize__  treedepth__  \
count         100.000000     100.000000  1.000000e+02   100.000000   
mean       -45757.879790       0.914038  1.173424e-02     8.430000   
std             7.014185       0.096583  3.486925e-18     0.639681   
min        -45774.243656       0.570781  1.173424e-02     6.000000   
25%        -45763.536779       0.882634  1.173424e-02     8.000000   
50%        -45756.786578       0.942247  1.173424e-02     8.500000   
75%        -45752.912315       0.983155  1.173424e-02     9.000000   
max        -45743.127666       0.999240  1.173424e-02     9.000000   

parameters  n_leapfrog__  divergent__      energy__        pi.1        pi.2  \
count         100.000000        100.0    100.000000  100.000000  100.000000   
mean          451.480000          0.0  45783.739820    0.033671    0.355802   
std           165.060655          0.0      9.128297    0.013258    0.045334   
min           127.000000          0.0  45765.058878  

## Training HMM Models (S=2 bis S=4)

Standard Hidden Markov Models mit konstanten Übergangswahrscheinlichkeiten.
(S=1 macht keinen Sinn für HMM und wird übersprungen)


In [108]:
# HMM Training für S=2 bis S=4
# WARNUNG: Dies kann mehrere Stunden dauern!

for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model(
            model_data=model_data,
            S=S,
            model_name="hmm",
            num_chains=NUM_CHAINS,
            num_samples=NUM_SAMPLES,
            num_warmup=NUM_WARMUP,
            seed=SEED,
        )

        trained_models[f"hmm_{S}"] = fit
        print(f"\n✓✓✓ HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("HMM Training Complete!")
print("=" * 60)



############################################################
# HMM Training: S=2
############################################################


Training HMM with S=2 states
Model file: ../data/stan_code/hmm.stan
Loading Stan model code...

Building and sampling model:
  Chains: 1
  Warmup iterations: 100
  Sampling iterations: 100
  Seed: 42
Building...

In file included from /Users/omidsedighi-mornani/Library/Caches/httpstan/4.13.0/models/4ks7zxfe/model_4ks7zxfe.cpp:2:
In file included from /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/httpstan/include/stan/model/model_header.hpp:4:
In file included from /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/httpstan/include/stan/math.hpp:19:
In file included from /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/httpstan/include/stan/math/rev.hpp:4:
In file included from /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/httpstan/include/stan/math/prim/fun/Eigen.hpp:23:
In file included from /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMA

5 warnings generated.

Building: 17.2s, done.Messages from stanc:
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_t5pz9wth/model_4ks7zxfe.stan', line 163, column 41: The
    variable state_emission may not have been assigned a value before its
    use.
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_t5pz9wth/model_4ks7zxfe.stan', line 120, column 33: The
    variable pos_obs may not have been assigned a value before its use.
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/200)
Sampling:  50% (100/200)
Sampling:  50% (101/200)
Sampling: 100% (200/200)
Sampling: 100% (200/200), done.
Messages received during sampling:
  Gradient evaluation took 0.179691 seconds
  1000 transitions using 10 leapfrog steps per transition would take 1796.91 seconds.
  Adjust your expectations accordingly!
           three stages of adaptation as currently configured.
     


✓ Model saved to ../models/hmm_2.pkl

------------------------------------------------------------
Model Summary:
------------------------------------------------------------

Sampled parameters: 2920 parameters, 100 draws
Parameter names (first 10): ['lp__', 'accept_stat__', 'stepsize__', 'treedepth__', 'n_leapfrog__', 'divergent__', 'energy__', 'pi.1', 'pi.2', 'tpm.1.1']

Basic statistics:


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in s

...

parameters          lp__  accept_stat__    stepsize__  treedepth__  \
count         100.000000     100.000000  1.000000e+02   100.000000   
mean       -45991.218863       0.893058  1.199439e-02     7.000000   
std             4.360200       0.118904  1.743463e-18     1.145919   
min        -46005.346086       0.392261  1.199439e-02     5.000000   
25%        -45993.562041       0.834400  1.199439e-02     6.000000   
50%        -45991.403781       0.944247  1.199439e-02     7.000000   
75%        -45988.616487       0.979741  1.199439e-02     8.000000   
max        -45978.872903       0.999876  1.199439e-02     9.000000   

parameters  n_leapfrog__  divergent__      energy__        pi.1        pi.2  \
count         100.000000        100.0    100.000000  100.000000  100.000000   
mean          233.560000          0.0  46009.588018    0.165627    0.504290   
std           173.077219          0.0      5.875781    0.021513    0.034790   
min            31.000000          0.0  45995.247417  


Building: found in cache, done.Messages from stanc:
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_t5pz9wth/model_4ks7zxfe.stan', line 163, column 41: The
    variable state_emission may not have been assigned a value before its
    use.
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_t5pz9wth/model_4ks7zxfe.stan', line 120, column 33: The
    variable pos_obs may not have been assigned a value before its use.
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%
Sampling:   0% (1/200)
Sampling:  50% (100/200)
Sampling:  50% (101/200)
Sampling: 100% (200/200)
Sampling: 100% (200/200), done.
Messages received during sampling:
  Gradient evaluation took 0.304888 seconds
  1000 transitions using 10 leapfrog steps per transition would take 3048.88 seconds.
  Adjust your expectations accordingly!
           three stages of adaptation as currently configured.
           Reducin


✓ Model saved to ../models/hmm_4.pkl

------------------------------------------------------------
Model Summary:
------------------------------------------------------------

Sampled parameters: 3970 parameters, 100 draws
Parameter names (first 10): ['lp__', 'accept_stat__', 'stepsize__', 'treedepth__', 'n_leapfrog__', 'divergent__', 'energy__', 'pi.1', 'pi.2', 'pi.3']

Basic statistics:


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in s

parameters          lp__  accept_stat__    stepsize__  treedepth__  \
count         100.000000     100.000000  1.000000e+02   100.000000   
mean       -45796.033031       0.887593  1.212738e-02     8.250000   
std             4.767990       0.130098  1.743463e-18     0.672324   
min        -45810.451844       0.399381  1.212738e-02     6.000000   
25%        -45799.548786       0.842180  1.212738e-02     8.000000   
50%        -45795.296267       0.944352  1.212738e-02     8.000000   
75%        -45792.550903       0.979750  1.212738e-02     9.000000   
max        -45787.844192       0.999108  1.212738e-02     9.000000   

parameters  n_leapfrog__  divergent__      energy__        pi.1        pi.2  \
count         100.000000        100.0    100.000000  100.000000  100.000000   
mean          425.880000          0.0  45819.813943    0.081036    0.381590   
std           208.637779          0.0      6.152532    0.018419    0.024869   
min           127.000000          0.0  45809.053042  

## Training Summary

Übersicht über alle trainierten Modelle


In [109]:
# Zusammenfassung aller trainierten Modelle
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"\nSaved in: {fitted_model_folder}")

# Alle gespeicherten Modell-Dateien auflisten
saved_models = sorted(fitted_model_folder.glob("*.pkl"))
print(f"\nSaved model files ({len(saved_models)}):")
for model_file in saved_models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name} ({size_mb:.2f} MB)")


TRAINING SUMMARY

Total models trained: 6
Models: ['vdhmm_2', 'vdhmm_3', 'vdhmm_4', 'hmm_2', 'hmm_3', 'hmm_4']

Saved in: ../models

Saved model files (6):
  - hmm_2.pkl (13.62 MB)
  - hmm_3.pkl (15.92 MB)
  - hmm_4.pkl (18.25 MB)
  - vdhmm_2.pkl (13.67 MB)
  - vdhmm_3.pkl (15.97 MB)
  - vdhmm_4.pkl (18.30 MB)


## Laden von trainierten Modellen

Falls du später ein bereits trainiertes Modell laden möchtest:


In [110]:
# Beispiel: Lade ein trainiertes Modell
def load_fitted_model(model_name, S):
    """
    Lädt ein bereits trainiertes Modell aus dem Dateisystem.

    Parameters:
    -----------
    model_name : str
        'vdhmm' oder 'hmm'
    S : int
        Anzahl der States

    Returns:
    --------
    dict : Dictionary mit fit-Objekt und Metadaten
    """
    model_path = fitted_model_folder / f"{model_name}_{S}.pkl"

    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")

    with open(model_path, "rb") as f:
        model_dict = pickle.load(f)

    print(f"✓ Loaded {model_name.upper()} with S={S}")
    print(f"  Keys: {list(model_dict.keys())}")

    # In PyStan 3: fit ist ein stan.fit.Fit Objekt
    fit = model_dict["fit"]
    print(f"  Type: {type(fit)}")

    # Beispiel: Zugriff auf Parameter
    df = fit.to_frame()
    print(f"  Parameters: {df.shape[1]} params, {df.shape[0]} draws")

    return model_dict


# Beispiel: Lade VD-HMM mit S=3
# loaded_model = load_fitted_model('vdhmm', 3)
# fit = loaded_model['fit']
#
# # Zugriff auf spezifischen Parameter (PyStan 3 Syntax)
# if 'omega_0' in fit.to_frame().columns:
#     omega_0_samples = fit['omega_0']  # Direkter Zugriff via Dictionary-Syntax
#     print(f"omega_0 mean: {omega_0_samples.mean()}")